# Atribuição de latitude e longitude aos hospitais (CNES)

Este notebook carrega os dados de estabelecimentos de saúde e a série diária de internações por hospital, faz o vínculo entre as bases e atribui **latitude** e **longitude** a cada estabelecimento com base no **CEP**.

## O que foi corrigido nesta versão

1. **Padronização de identificadores** antes do `merge` (`CGC_HOSP`, `CPF_CNPJ` e `COD_CEP`).
2. **Deduplicação do cadastro de estabelecimentos** por CNPJ, evitando explosão de linhas no `merge`.
3. **Geocodificação por CEP único**, em vez de consultar a API repetidamente para cada linha da série diária.
4. **Fallback de geocodificação**:
   - primeiro tenta a **BrasilAPI CEP v2**;
   - se não vier coordenada, usa o endereço retornado para consultar o **Nominatim**.
5. Inclusão de **logs, resumo de cobertura** e **salvamento dos resultados**.

> **Observação importante:** o Nominatim público limita o uso a, no máximo, **1 requisição por segundo** e desencoraja geocodificação em lote sem cache. Por isso, este notebook consulta apenas **CEPs únicos** e usa a BrasilAPI como primeira tentativa. citeturn753866view1turn485820search0


## Importações e configurações


In [18]:
import os
import re
import time
import logging

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)


In [19]:
ANOS_ESTABELECIMENTOS = [2012, 2013, 2014, 2015, 2016, 2017, 2018]

URL_TEMPLATE_ESTABELECIMENTOS = (
    "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/"
    "refs/heads/Refactoring-And-Documentation/Data/RawData/DataSus/cnes/"
    "estabelecimento_saude_rj_{ano}.parquet"
)

URL_HOSPITAIS = (
    "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/"
    "refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/DataSus/"
    "respiratory_hospitalization_time_series_by_hospital.csv"
)

OUTPUT_DIR = "output"
OUTPUT_PARQUET = os.path.join(OUTPUT_DIR, "hospitais_cnes_lat_lon.parquet")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "hospitais_cnes_lat_lon.csv")

USER_AGENT = "QualiAr-Geocoder/1.0 (academic research project; single-threaded; contact: update-me)"
TIMEOUT = 20
NOMINATIM_DELAY = 1.1  # respeita a política pública de ~1 req/s


## Funções auxiliares


In [3]:
def limpar_documento(value, tamanho=None):
    """Remove tudo que não for dígito e, se solicitado, preenche à esquerda com zeros."""
    if pd.isna(value):
        return pd.NA

    if isinstance(value, float) and value.is_integer():
        value = int(value)

    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]

    digits = re.sub(r"\D", "", text)

    if not digits:
        return pd.NA

    if tamanho is not None:
        if len(digits) > tamanho:
            return pd.NA
        digits = digits.zfill(tamanho)

    return digits


def limpar_cep(value):
    """Padroniza o CEP em 8 dígitos. Retorna NA se o valor for inválido."""
    return limpar_documento(value, tamanho=8)


def limpar_cnpj(value):
    """Padroniza CNPJ/CPF_CNPJ em 14 dígitos."""
    return limpar_documento(value, tamanho=14)


def criar_sessao():
    """Cria uma sessão HTTP com tentativas automáticas para erros transitórios."""
    session = requests.Session()

    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": USER_AGENT})

    return session


SESSION = criar_sessao()


def safe_float(value):
    try:
        if value in (None, "", pd.NA):
            return None
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return None


## Carregamento dos dados


In [4]:
df_estabelecimentos = pd.DataFrame()

for ano in ANOS_ESTABELECIMENTOS:
    url = URL_TEMPLATE_ESTABELECIMENTOS.format(ano=ano)
    df_ano = pd.read_parquet(url)
    df_ano["ANO_REF"] = ano
    df_estabelecimentos = pd.concat([df_estabelecimentos, df_ano], ignore_index=True)
    logging.info(f"Dados de estabelecimentos do ano {ano} carregados com sucesso.")

print("Dimensão consolidada de estabelecimentos:", df_estabelecimentos.shape)
display(df_estabelecimentos.head())


2026-04-01 14:35:02,533 | INFO | Dados de estabelecimentos do ano 2012 carregados com sucesso.
2026-04-01 14:35:02,966 | INFO | Dados de estabelecimentos do ano 2013 carregados com sucesso.
2026-04-01 14:35:03,408 | INFO | Dados de estabelecimentos do ano 2014 carregados com sucesso.
2026-04-01 14:35:03,890 | INFO | Dados de estabelecimentos do ano 2015 carregados com sucesso.
2026-04-01 14:35:05,369 | INFO | Dados de estabelecimentos do ano 2016 carregados com sucesso.
2026-04-01 14:35:05,958 | INFO | Dados de estabelecimentos do ano 2017 carregados com sucesso.
2026-04-01 14:35:07,579 | INFO | Dados de estabelecimentos do ano 2018 carregados com sucesso.


Dimensão consolidada de estabelecimentos: (686201, 6)


,CNES,CODUFMUN,COD_CEP,CPF_CNPJ,COD_IR,ANO_REF
0,2288354,330455,20745150,00000000000000,10,2012
1,2280221,330455,22241010,42144253000189,,2012
2,2295857,330170,25030120,00000000000000,10,2012
3,2277271,330455,20941160,29468055000102,,2012
4,2271257,330455,21032000,28000107000744,,2012


In [20]:
df_hospitais = pd.read_csv(URL_HOSPITAIS)

print("Dimensão da série de internações:", df_hospitais.shape)
display(df_hospitais.head())


Dimensão da série de internações: (163648, 5)


,CGC_HOSP,data_dia,num_internacoes,CNES,COD_CEP
0,254512000184,2012-01-01,28,2296748,26112140
1,254512000184,2012-01-02,98,2296748,26112140
2,254512000184,2012-01-03,0,2296748,26112140
3,254512000184,2012-01-04,14,2296748,26112140
4,254512000184,2012-01-05,28,2296748,26112140


## Padronização das chaves e vínculo entre as bases

Nesta etapa:

- padronizamos `CGC_HOSP` e `CPF_CNPJ` em **14 dígitos**;
- padronizamos `COD_CEP` em **8 dígitos**;
- deduplicamos o cadastro de estabelecimentos por `CPF_CNPJ`, priorizando registros mais recentes com CEP válido;
- realizamos o `merge` entre a série diária e o cadastro tratado.


In [6]:
df_hospitais["CGC_HOSP"] = df_hospitais["CGC_HOSP"].apply(limpar_cnpj)

cadastro_estabelecimentos = df_estabelecimentos.copy()
cadastro_estabelecimentos["CPF_CNPJ"] = cadastro_estabelecimentos["CPF_CNPJ"].apply(limpar_cnpj)
cadastro_estabelecimentos["COD_CEP"] = cadastro_estabelecimentos["COD_CEP"].apply(limpar_cep)

cadastro_estabelecimentos["TEM_CEP_VALIDO"] = cadastro_estabelecimentos["COD_CEP"].notna().astype(int)

cadastro_estabelecimentos = (
    cadastro_estabelecimentos
    .sort_values(["CPF_CNPJ", "TEM_CEP_VALIDO", "ANO_REF"])
    .drop_duplicates(subset=["CPF_CNPJ"], keep="last")
    .drop(columns=["TEM_CEP_VALIDO"])
)

df_hospitais = df_hospitais.merge(
    cadastro_estabelecimentos[["CNES", "CPF_CNPJ", "COD_CEP"]],
    left_on="CGC_HOSP",
    right_on="CPF_CNPJ",
    how="left"
).drop(columns=["CPF_CNPJ"])


In [7]:
print("Quantidade de linhas após o merge:", len(df_hospitais))
print("Quantidade de hospitais sem correspondência de CNES:", df_hospitais["CNES"].isna().sum())
print("Quantidade de linhas sem CEP válido:", df_hospitais["COD_CEP"].isna().sum())

display(df_hospitais.head())
display(df_hospitais["COD_CEP"].dropna().head(10))


Quantidade de linhas após o merge: 306840
Quantidade de hospitais sem correspondência de CNES: 143192
Quantidade de linhas sem CEP válido: 143192


,CGC_HOSP,data_dia,num_internacoes,CNES,COD_CEP
0,00254512000184,2012-01-01,28,2296748,26112140
1,00254512000184,2012-01-02,98,2296748,26112140
2,00254512000184,2012-01-03,0,2296748,26112140
3,00254512000184,2012-01-04,14,2296748,26112140
4,00254512000184,2012-01-05,28,2296748,26112140


0    26112140
1    26112140
2    26112140
3    26112140
4    26112140
5    26112140
6    26112140
7    26112140
8    26112140
9    26112140
Name: COD_CEP, dtype: object

In [13]:
# Vendo total de unidades hospitalares distintas
total_unidades_hospitalares = df_hospitais["CGC_HOSP"].nunique()
print(f"Total de unidades hospitalares distintas: {total_unidades_hospitalares}")

# Vendo total de unidades hispitalares distintas com CNES
total_unidades_hospitalares_com_cnes = df_hospitais["CNES"].nunique()
print(f"Total de unidades hospitalares distintas com CNES: {total_unidades_hospitalares_com_cnes}")

# Removendo hospitais sem CNES
df_hospitais.dropna(subset=["CNES"], inplace=True)

# Quantidade de linhas e hospitais distintos após remoção de hospitais sem CNES
print("Dimensão do DataFrame após remoção de hospitais sem CNES:", df_hospitais.shape)
print("Total de unidades hospitalares distintas após remoção de hospitais sem CNES:", df_hospitais["CGC_HOSP"].nunique())
print("Total de unidades hospitalares distintas com CNES após remoção de hospitais sem CNES:", df_hospitais["CNES"].nunique())

Total de unidades hospitalares distintas: 120
Total de unidades hospitalares distintas com CNES: 64
Dimensão do DataFrame após remoção de hospitais sem CNES: (163648, 5)
Total de unidades hospitalares distintas após remoção de hospitais sem CNES: 64
Total de unidades hospitalares distintas com CNES após remoção de hospitais sem CNES: 64


## Geocodificação dos CEPs

### Estratégia adotada

1. Tenta obter coordenadas diretamente pela **BrasilAPI CEP v2**.
2. Se a BrasilAPI não devolver latitude/longitude, consulta o **ViaCEP** para obter um endereço textual mais confiável.
3. Com esse endereço, tenta o **Nominatim** em múltiplas consultas, do formato mais específico ao mais simples.
4. Cada **CEP é consultado apenas uma vez** e o resultado é reaproveitado no `merge` final.

### Observações

- A geocodificação por **CEP puro** costuma ter cobertura baixa.
- Por isso, o notebook prioriza o fluxo **CEP → endereço → latitude/longitude**.
- O Nominatim público é usado apenas como *fallback* e com atraso entre requisições.


In [14]:
def consultar_brasilapi_cep(cep):
    """
    Consulta o CEP na BrasilAPI v2 e retorna endereço + coordenadas (quando disponíveis).

    Returns
    -------
    dict | None
        Dicionário padronizado com dados do endereço e possíveis coordenadas.
    """
    url = f"https://brasilapi.com.br/api/cep/v2/{cep}"

    response = SESSION.get(url, timeout=TIMEOUT)

    if response.status_code == 404:
        return None

    response.raise_for_status()
    data = response.json()

    location = data.get("location") or {}
    coordinates = location.get("coordinates") or {}

    return {
        "cep": cep,
        "street": data.get("street"),
        "neighborhood": data.get("neighborhood"),
        "city": data.get("city"),
        "state": data.get("state"),
        "lat": safe_float(coordinates.get("latitude")),
        "lon": safe_float(coordinates.get("longitude")),
        "fonte_endereco": "BrasilAPI v2",
    }


def consultar_viacep(cep):
    """
    Consulta o CEP no ViaCEP e retorna o endereço textual.

    Returns
    -------
    dict | None
        Dicionário padronizado com dados do endereço, sem coordenadas.
    """
    url = f"https://viacep.com.br/ws/{cep}/json/"

    response = SESSION.get(url, timeout=TIMEOUT)

    if response.status_code == 400:
        return None

    response.raise_for_status()
    data = response.json()

    if data.get("erro"):
        return None

    return {
        "cep": cep,
        "street": data.get("logradouro"),
        "neighborhood": data.get("bairro"),
        "city": data.get("localidade"),
        "state": data.get("uf"),
        "lat": None,
        "lon": None,
        "fonte_endereco": "ViaCEP",
    }


def normalizar_parte_endereco(value):
    """Normaliza partes do endereço para montagem das consultas textuais."""
    if value is None or pd.isna(value):
        return None

    text = str(value).strip()
    if not text:
        return None

    return text


def montar_consultas_textuais(cep, dados_endereco):
    """
    Monta uma lista de consultas textuais para o Nominatim, da mais específica à mais ampla.

    A ideia é aumentar a cobertura sem depender apenas de uma única string de busca.
    """
    if not dados_endereco:
        return [f"CEP {cep}, Brasil"] if cep else []

    street = normalizar_parte_endereco(dados_endereco.get("street"))
    neighborhood = normalizar_parte_endereco(dados_endereco.get("neighborhood"))
    city = normalizar_parte_endereco(dados_endereco.get("city"))
    state = normalizar_parte_endereco(dados_endereco.get("state"))

    consultas = []

    combinacoes = [
        [street, neighborhood, city, state, "Brasil"],
        [street, city, state, "Brasil"],
        [neighborhood, city, state, "Brasil"],
        [city, state, "Brasil"],
        [f"CEP {cep}", city, state, "Brasil"] if city or state else [f"CEP {cep}", "Brasil"],
        [f"{cep}", "Brasil"],
    ]

    for partes in combinacoes:
        partes_validas = [p for p in partes if p]
        if not partes_validas:
            continue

        query = ", ".join(partes_validas)
        if query not in consultas:
            consultas.append(query)

    return consultas


def consultar_nominatim(query):
    """
    Consulta o Nominatim a partir de uma string textual de endereço.

    Returns
    -------
    tuple
        (latitude, longitude) ou (None, None)
    """
    if not query:
        return (None, None)

    time.sleep(NOMINATIM_DELAY)

    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "jsonv2",
        "limit": 1,
        "countrycodes": "br",
        "addressdetails": 1,
    }

    response = SESSION.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    if not data:
        return (None, None)

    return safe_float(data[0].get("lat")), safe_float(data[0].get("lon"))


def geocodificar_cep(cep):
    """
    Geocodifica um CEP usando a seguinte ordem de prioridade:

    1. BrasilAPI v2 com coordenadas prontas.
    2. ViaCEP para obter endereço textual.
    3. Nominatim com múltiplas consultas textuais como fallback.

    Returns
    -------
    dict
        Resultado padronizado com coordenadas, fonte e consulta utilizada.
    """
    cep = limpar_cep(cep)

    if pd.isna(cep):
        return {
            "COD_CEP": pd.NA,
            "LAT": None,
            "LNG": None,
            "FONTE_GEOCODIFICACAO": "CEP inválido",
            "FONTE_ENDERECO": None,
            "QUERY_UTILIZADA": None,
        }

    dados_brasilapi = None
    dados_viacep = None

    try:
        dados_brasilapi = consultar_brasilapi_cep(cep)
    except Exception as exc:
        logging.warning(f"Falha na BrasilAPI para o CEP {cep}: {exc}")

    if (
        dados_brasilapi
        and dados_brasilapi.get("lat") is not None
        and dados_brasilapi.get("lon") is not None
    ):
        return {
            "COD_CEP": cep,
            "LAT": dados_brasilapi["lat"],
            "LNG": dados_brasilapi["lon"],
            "FONTE_GEOCODIFICACAO": "BrasilAPI v2",
            "FONTE_ENDERECO": dados_brasilapi.get("fonte_endereco"),
            "QUERY_UTILIZADA": None,
        }

    try:
        dados_viacep = consultar_viacep(cep)
    except Exception as exc:
        logging.warning(f"Falha no ViaCEP para o CEP {cep}: {exc}")

    candidatos_endereco = []
    if dados_viacep:
        candidatos_endereco.append(dados_viacep)
    if dados_brasilapi:
        candidatos_endereco.append(dados_brasilapi)

    for dados_endereco in candidatos_endereco:
        consultas = montar_consultas_textuais(cep, dados_endereco)

        for query in consultas:
            try:
                lat, lon = consultar_nominatim(query)

                if lat is not None and lon is not None:
                    return {
                        "COD_CEP": cep,
                        "LAT": lat,
                        "LNG": lon,
                        "FONTE_GEOCODIFICACAO": "ViaCEP + Nominatim"
                        if dados_endereco.get("fonte_endereco") == "ViaCEP"
                        else "BrasilAPI + Nominatim",
                        "FONTE_ENDERECO": dados_endereco.get("fonte_endereco"),
                        "QUERY_UTILIZADA": query,
                    }
            except Exception as exc:
                logging.warning(
                    f"Falha no Nominatim para o CEP {cep} usando a consulta '{query}': {exc}"
                )

    return {
        "COD_CEP": cep,
        "LAT": None,
        "LNG": None,
        "FONTE_GEOCODIFICACAO": "Não encontrado",
        "FONTE_ENDERECO": (
            dados_viacep.get("fonte_endereco")
            if dados_viacep
            else dados_brasilapi.get("fonte_endereco")
            if dados_brasilapi
            else None
        ),
        "QUERY_UTILIZADA": None,
    }

In [15]:
ceps_unicos = (
    df_hospitais["COD_CEP"]
    .dropna()
    .apply(limpar_cep)
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Quantidade de CEPs únicos válidos a geocodificar:", len(ceps_unicos))

resultados_geocodificacao = []
for i, cep in enumerate(tqdm(ceps_unicos, desc="Geocodificando CEPs únicos"), start=1):
    resultados_geocodificacao.append(geocodificar_cep(cep))

    if i % 100 == 0:
        logging.info(f"CEPs processados: {i} / {len(ceps_unicos)}")

df_ceps_geo = pd.DataFrame(resultados_geocodificacao)

print("Resumo da geocodificação por fonte:")
display(df_ceps_geo["FONTE_GEOCODIFICACAO"].value_counts(dropna=False))
display(df_ceps_geo.head())

Quantidade de CEPs únicos válidos a geocodificar: 63


Geocodificando CEPs únicos: 100%|██████████| 63/63 [01:36<00:00,  1.53s/it]

Resumo da geocodificação por fonte:


FONTE_GEOCODIFICACAO
ViaCEP + Nominatim    57
BrasilAPI v2           6
Name: count, dtype: int64

,COD_CEP,LAT,LNG,FONTE_GEOCODIFICACAO,FONTE_ENDERECO,QUERY_UTILIZADA
0,20020021,-22.909003,-43.171766,ViaCEP + Nominatim,ViaCEP,"Rua Santa Luzia, Centro, Rio de Janeiro, RJ, B..."
1,20211270,-22.913540,-43.203699,ViaCEP + Nominatim,ViaCEP,"Rua Estácio de Sá, Estácio, Rio de Janeiro, RJ..."
2,20211350,-22.906713,-43.188673,ViaCEP + Nominatim,ViaCEP,"Praça da República, Centro, Rio de Janeiro, RJ..."
3,20221160,-22.897271,-43.182278,ViaCEP + Nominatim,ViaCEP,"Rua Sacadura Cabral, Saúde, Rio de Janeiro, RJ..."
4,20261064,-22.922985,-43.213625,ViaCEP + Nominatim,ViaCEP,"Rua do Bispo, Rio Comprido, Rio de Janeiro, RJ..."


In [16]:
df_hospitais = df_hospitais.drop(
    columns=["LAT", "LNG", "FONTE_GEOCODIFICACAO", "FONTE_ENDERECO", "QUERY_UTILIZADA"],
    errors="ignore"
)

df_hospitais = df_hospitais.merge(
    df_ceps_geo,
    on="COD_CEP",
    how="left"
)

df_hospitais_final = df_hospitais.copy()

cobertura = df_hospitais_final["LAT"].notna().mean() * 100
print(f"Cobertura de geocodificação na base final: {cobertura:.2f}%")

display(
    df_hospitais_final[
        ["CNES", "CGC_HOSP", "COD_CEP", "LAT", "LNG", "FONTE_GEOCODIFICACAO", "FONTE_ENDERECO"]
    ].head()
)

display(df_hospitais_final["FONTE_GEOCODIFICACAO"].value_counts(dropna=False))

Cobertura de geocodificação na base final: 100.00%


,CNES,CGC_HOSP,COD_CEP,LAT,LNG,FONTE_GEOCODIFICACAO,FONTE_ENDERECO
0,2296748,00254512000184,26112140,-22.748938,-43.410039,ViaCEP + Nominatim,ViaCEP
1,2296748,00254512000184,26112140,-22.748938,-43.410039,ViaCEP + Nominatim,ViaCEP
2,2296748,00254512000184,26112140,-22.748938,-43.410039,ViaCEP + Nominatim,ViaCEP
3,2296748,00254512000184,26112140,-22.748938,-43.410039,ViaCEP + Nominatim,ViaCEP
4,2296748,00254512000184,26112140,-22.748938,-43.410039,ViaCEP + Nominatim,ViaCEP


FONTE_GEOCODIFICACAO
ViaCEP + Nominatim    148306
BrasilAPI v2           15342
Name: count, dtype: int64

In [17]:
df_hospitais_final.tail()

,CGC_HOSP,data_dia,num_internacoes,CNES,COD_CEP,LAT,LNG,FONTE_GEOCODIFICACAO,FONTE_ENDERECO,QUERY_UTILIZADA
163643,53221255004995,2018-12-27,0,7065515,20530001,-22.927102,-43.235785,ViaCEP + Nominatim,ViaCEP,"Rua Conde de Bonfim, Tijuca, Rio de Janeiro, R..."
163644,53221255004995,2018-12-28,0,7065515,20530001,-22.927102,-43.235785,ViaCEP + Nominatim,ViaCEP,"Rua Conde de Bonfim, Tijuca, Rio de Janeiro, R..."
163645,53221255004995,2018-12-29,0,7065515,20530001,-22.927102,-43.235785,ViaCEP + Nominatim,ViaCEP,"Rua Conde de Bonfim, Tijuca, Rio de Janeiro, R..."
163646,53221255004995,2018-12-30,0,7065515,20530001,-22.927102,-43.235785,ViaCEP + Nominatim,ViaCEP,"Rua Conde de Bonfim, Tijuca, Rio de Janeiro, R..."
163647,53221255004995,2018-12-31,0,7065515,20530001,-22.927102,-43.235785,ViaCEP + Nominatim,ViaCEP,"Rua Conde de Bonfim, Tijuca, Rio de Janeiro, R..."


## Salvamento dos resultados


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_hospitais_final.to_parquet(OUTPUT_PARQUET, index=False)
df_hospitais_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Arquivo Parquet salvo em: {OUTPUT_PARQUET}")
print(f"Arquivo CSV salvo em: {OUTPUT_CSV}")
